In [0]:
storage_key = dbutils.secrets.get(scope="kv-finbank", key="storage-account-key")
spark.conf.set("fs.azure.account.key.stfinbankdevfbcq2026.dfs.core.windows.net",storage_key)

In [0]:
table_name = dbutils.widgets.get("table_name")
tipo_carga = dbutils.widgets.get("tipo_carga")
columna_control = dbutils.widgets.get("columna_control")
rows_copied = int(dbutils.widgets.get("rows_copied"))
duration = int(dbutils.widgets.get("duration"))
data_written = int(float(dbutils.widgets.get("data_written"))) 
run_id = dbutils.widgets.get("run_id")
year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")
day = dbutils.widgets.get("day")

columna_control_valida = columna_control not in (None, "", "null", "None")
 
print(f"procesando: {table_name}, tipo={tipo_carga},filas={rows_copied},columna_control={columna_control_valida}")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import max as spark_max
 
control_path = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net/_control/control_ingestas"
log_path = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net/_control/log_ingestas"
 
if tipo_carga == "Delta" and rows_copied > 0 and columna_control_valida:
    bronze_partition_path = (
        f"abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net/"
        f"{table_name}/{year}/{month}/{day}/"
    )
    df_bronze_act = spark.read.parquet(bronze_partition_path).filter(f"batch_id = '{run_id}'")
 
    from pyspark.sql.functions import col, current_timestamp
    df_bronze_valido = df_bronze_act.filter(col(columna_control) <= current_timestamp())
 
    checkpoint_act = df_bronze_valido.agg(spark_max(columna_control).alias("mx")).collect()[0]["mx"]
 
    if checkpoint_act is not None:
        control_table = DeltaTable.forPath(spark, control_path)
        control_table.update(
            condition=f"nombre_tabla = '{table_name}'",
            set={"checkpoint_date": f"'{checkpoint_act}'"}
        )
        print(f"checkpoint actualizado para {table_name}")
    else:
        print(f"checkpoint sin cambios")
else:
    print(f"Es full, o sin filas nuevas)")

In [0]:
from pyspark.sql.functions import col as _col
from datetime import datetime, timezone

ALERTAS_VOLUMEN_PATH = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net/_control/alertas_volumen"

df_historial = (
    spark.read.format("delta").load(log_path)
    .filter((_col("nombre_tabla") == table_name) & (_col("estado") == "Succeeded"))
    .orderBy(_col("fecha_ejecucion").desc())
    .limit(7)
)
filas_historial = [r["filas_copiadas"] for r in df_historial.collect() if r["filas_copiadas"] is not None]

if len(filas_historial) > 0:
    promedio_previo = sum(filas_historial) / len(filas_historial)

    if promedio_previo > 0:
        pct_diferencia = abs(rows_copied - promedio_previo) / promedio_previo * 100

        if pct_diferencia > 30:
            from pyspark.sql import Row
            from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

            schema_alerta = StructType([
                StructField("nombre_tabla", StringType(), False),
                StructField("filas_actuales", IntegerType(), False),
                StructField("promedio_historico", DoubleType(), False),
                StructField("pct_diferencia", DoubleType(), False),
                StructField("fecha_deteccion", TimestampType(), False),
                StructField("run_id", StringType(), False),
            ])
            fila_alerta = spark.createDataFrame(
                [Row(
                    nombre_tabla=table_name,
                    filas_actuales=rows_copied,
                    promedio_historico=round(promedio_previo, 2),
                    pct_diferencia=round(pct_diferencia, 2),
                    fecha_deteccion=datetime.now(timezone.utc),
                    run_id=run_id,
                )],
                schema=schema_alerta
            )
            fila_alerta.write.format("delta").mode("append").option("mergeSchema", "true").save(ALERTAS_VOLUMEN_PATH)

            raise Exception(
                f"anomalia en el volumen: {table_name} tuvo {rows_copied} filas "
                f"mientra que el promedio historico de {round(promedio_previo,2)} "
                f"{round(pct_diferencia,2)}% de diferencia, supera el 30% permitido"
            )
        else:
            print(f"volumen normal {rows_copied} filas vs. promedio historico {round(promedio_previo,2)} {round(pct_diferencia,2)}% de diferencia")
    else:
        print(f"sin historial previo suficiente para comparar volumen de {table_name}")
else:
    print(f" primera ejecucion registrada para {table_name}, sin historial para comparar")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType

schema_log = StructType([
    StructField("nombre_tabla", StringType(), False),
    StructField("fecha_ejecucion", TimestampType(), False),
    StructField("filas_copiadas", IntegerType(), True),
    StructField("duracion_segundos", IntegerType(), True),
    StructField("tamano_bytes", IntegerType(), True),
    StructField("estado", StringType(), False),
    StructField("run_id", StringType(), False),
])

nueva_fila_log = spark.createDataFrame(
    [Row(
        nombre_tabla=table_name,
        fecha_ejecucion=datetime.now(timezone.utc),
        filas_copiadas=rows_copied,
        duracion_segundos=duration,
        tamano_bytes=data_written,
        estado="Succeeded",
        run_id=run_id,
    )],
    schema=schema_log
)
nueva_fila_log.write.format("delta").mode("append").option("mergeSchema", "true").save(log_path)
print(f"Log registrado para {table_name}")

In [0]:
dbutils.notebook.exit("OK")